# 🎬 «Пляж» — короткометражка в стиле Arcane
### Рендер на бесплатном GPU (Colab / Kaggle) через ComfyUI

**Хронометраж:** ~5:11 · **59 планов** · 1920×1080 · 24 fps

Порядок работы: ячейки идут сверху вниз. Тяжёлые этапы (кадры → лица → апскейл → видео)
сохраняют промежуточные файлы на Google Drive, поэтому при обрыве сессии
можно продолжить с того же места — всё уже готовое пропускается.

**Политика контента:** 18+, но БЕЗ откровенных сцен. Интимные планы решаются
силуэтами, тёмной водой и тенью. Реальные узнаваемые лица не используются
для откровенных изображений. Это зашито в промпты и негативы.

> Runtime → Change runtime type → **GPU (T4)**

## 1. Проверка GPU

In [ ]:
!nvidia-smi
import torch, sys
print('torch', torch.__version__, '| cuda', torch.cuda.is_available())
if torch.cuda.is_available():
    p = torch.cuda.get_device_properties(0)
    print(f'GPU: {p.name} | VRAM: {p.total_memory/1024**3:.1f} GB')
    if p.total_memory/1024**3 < 14:
        print('ВНИМАНИЕ: мало VRAM — используйте SDXL вместо FLUX и LTX вместо Wan.')
else:
    sys.exit('GPU не выдан. Runtime -> Change runtime type -> GPU')

## 2. Google Drive (сохранение прогресса)

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import os
PROJ = '/content/drive/MyDrive/PLYAZH_ARCANE'
# Публичны только служебные JSON из репозитория; фото всегда остаются на Drive.
ASSET_BASE_URL = 'https://raw.githubusercontent.com/Ejonnyss/film/main/PRODUCTION/11_CLOUD'
DIRS = {
    'refs':      f'{PROJ}/00_refs',
    'keyframes': f'{PROJ}/01_keyframes',
    'faces':     f'{PROJ}/02_faces',
    'upscaled':  f'{PROJ}/03_upscaled',
    'clips':     f'{PROJ}/04_clips',
    'audio':     f'{PROJ}/05_audio',
    'delivery':  f'{PROJ}/06_delivery',
}
for d in DIRS.values():
    os.makedirs(d, exist_ok=True)
print('Рабочая папка:', PROJ)
for k, v in DIRS.items():
    print(f'  {k:10s} {len(os.listdir(v)):3d} файлов')

## 3. Установка ComfyUI и нод
Версии зафиксированы на проверенных commit SHA. Повторный запуск безопасен.

In [ ]:
import os, pathlib, subprocess, sys

COMFY_COMMIT = '83082a51c420a364b15ea5f40d61da74e35b2da5'
NODE_PINS = {
    'ComfyUI_InstantID': (
        'https://github.com/cubiq/ComfyUI_InstantID.git',
        '72495e806bc2ab9c41581e15ccaa1bcf83c477e8'),
    'ComfyUI-VideoHelperSuite': (
        'https://github.com/Kosinkadink/ComfyUI-VideoHelperSuite.git',
        '4ee72c065db22c9d96c2427954dc69e7b908444b'),
}

def run(cmd, cwd=None):
    print('+', ' '.join(map(str, cmd)))
    subprocess.run(list(map(str, cmd)), cwd=cwd, check=True)

def ensure_repo(url, path, commit):
    path = pathlib.Path(path)
    if not (path / '.git').exists():
        run(['git','clone','--filter=blob:none','--no-checkout',url,path])
    run(['git','fetch','--depth','1','origin',commit], cwd=path)
    run(['git','checkout','--detach',commit], cwd=path)
    got = subprocess.check_output(['git','rev-parse','HEAD'], cwd=path, text=True).strip()
    assert got == commit, (path, got, commit)
    return got

ensure_repo('https://github.com/Comfy-Org/ComfyUI.git', '/content/ComfyUI', COMFY_COMMIT)
run([sys.executable,'-m','pip','install','-q','-r','/content/ComfyUI/requirements.txt'])

node_root = pathlib.Path('/content/ComfyUI/custom_nodes')
node_root.mkdir(parents=True, exist_ok=True)
for name, (url, commit) in NODE_PINS.items():
    path = node_root / name
    ensure_repo(url, path, commit)
    req = path / 'requirements.txt'
    if req.exists():
        run([sys.executable,'-m','pip','install','-q','-r',req])

run([sys.executable,'-m','pip','install','-q',
     'insightface==0.7.3','onnxruntime-gpu','opencv-python-headless','imageio-ffmpeg'])
print('ComfyUI:', COMFY_COMMIT)
print('Custom nodes:', {k:v[1] for k,v in NODE_PINS.items()})

## 4. Загрузка моделей
SDXL + Arcane-LoRA для кадров, InstantID для лиц, 4x-UltraSharp для
апскейла и LTX-Video 0.9.5 + T5XXL для анимации. Размер и SHA-256
проверяются; повреждённый кэш перескачивается.

In [ ]:
import hashlib, os, pathlib, subprocess
M = '/content/ComfyUI/models'
for sub in ['checkpoints','loras','instantid','controlnet','insightface/models/antelopev2',
            'clip_vision','upscale_models','diffusion_models','vae','text_encoders']:
    os.makedirs(f'{M}/{sub}', exist_ok=True)

def sha256(path, block=16*1024*1024):
    h = hashlib.sha256()
    with open(path, 'rb') as f:
        for chunk in iter(lambda: f.read(block), b''):
            h.update(chunk)
    return h.hexdigest()

def fetch(spec):
    path = pathlib.Path(spec['path'])
    path.parent.mkdir(parents=True, exist_ok=True)
    def valid():
        return (path.exists() and path.stat().st_size == spec['size'] and
                sha256(path) == spec['sha256'])
    if valid():
        print('OK cache:', path.name); return
    if path.exists():
        print('invalid cache, deleting:', path)
        path.unlink()
    subprocess.run(['wget','-c','--tries=5','--timeout=30','--show-progress',
                    '-O',str(path),spec['url']], check=True)
    if not valid():
        raise RuntimeError(f'checksum/size mismatch: {path}')
    print('verified:', path.name)

MODEL_SPECS = [
 {'path':f'{M}/checkpoints/sd_xl_base_1.0.safetensors','size':6938078334,
  'sha256':'31e35c80fc4829d14f90153f4c74cd59c90b779f6afe05a74cd6120b893f7e5b',
  'url':'https://huggingface.co/stabilityai/stable-diffusion-xl-base-1.0/resolve/main/sd_xl_base_1.0.safetensors'},
 {'path':f'{M}/loras/arcane_style_xl_v1.safetensors','size':170537868,
  'sha256':'23767b60d4aa1a1e49fd053f1ae14d150d9a2b1bff2c5a1fb1cdb565ce2afb1f',
  'url':'https://civitai.com/api/download/models/211361'},
 {'path':f'{M}/instantid/ip-adapter.bin','size':1691134141,
  'sha256':'02b3618e36d803784166660520098089a81388e61a93ef8002aa79a5b1c546e1',
  'url':'https://huggingface.co/InstantX/InstantID/resolve/main/ip-adapter.bin'},
 {'path':f'{M}/controlnet/instantid_controlnet.safetensors','size':2502139136,
  'sha256':'c8127be9f174101ebdafee9964d856b49b634435cf6daa396d3f593cf0bbbb05',
  'url':'https://huggingface.co/InstantX/InstantID/resolve/main/ControlNetModel/diffusion_pytorch_model.safetensors'},
 {'path':f'{M}/upscale_models/4x-UltraSharp.pth','size':66961958,
  'sha256':'a5812231fc936b42af08a5edba784195495d303d5b3248c24489ef0c4021fe01',
  'url':'https://huggingface.co/uwg/upscaler/resolve/main/ESRGAN/4x-UltraSharp.pth'},
 {'path':f'{M}/checkpoints/ltx-video-2b-v0.9.5.safetensors','size':6340729500,
  'sha256':'720d15c9f19f7d0f6b2a92bbbc34410e2cfb2f6856a100b38f734fbf973d4adf',
  'url':'https://huggingface.co/Lightricks/LTX-Video/resolve/main/ltx-video-2b-v0.9.5.safetensors'},
 {'path':f'{M}/text_encoders/t5xxl_fp16.safetensors','size':9787841024,
  'sha256':'6e480b09fae049a72d2a8c5fbccb8d3e92febeb233bbe9dfe7256958a9167635',
  'url':'https://huggingface.co/comfyanonymous/flux_text_encoders/resolve/main/t5xxl_fp16.safetensors'},
]
for spec in MODEL_SPECS:
    fetch(spec)
print('All model files verified')

### 4b. Antelopev2 для InstantID
Пакет детекции лиц — без него InstantID не стартует.

In [ ]:
import pathlib, shutil, subprocess
zip_path = pathlib.Path('/content/antelopev2.zip')
antelope = pathlib.Path('/content/ComfyUI/models/insightface/models/antelopev2')
expected = {'1k3d68.onnx','2d106det.onnx','genderage.onnx','glintr100.onnx','scrfd_10g_bnkps.onnx'}
if not expected.issubset({p.name for p in antelope.glob('*.onnx')}):
    spec = {'path':str(zip_path),'size':360662982,
            'sha256':'8e182f14fc6e80b3bfa375b33eb6cff7ee05d8ef7633e738d1c89021dcf0c5c5',
            'url':'https://huggingface.co/MonsterMMORPG/tools/resolve/main/antelopev2.zip'}
    fetch(spec)
    subprocess.run(['unzip','-q','-o',str(zip_path),'-d',str(antelope.parent)], check=True)
found = {p.name for p in antelope.glob('*.onnx')}
missing = expected - found
if missing:
    raise RuntimeError(f'antelopev2 incomplete: {sorted(missing)}')
print('antelopev2 OK:', sorted(found))

## 5. Референсы лиц
Положите на Drive в `PLYAZH_ARCANE/00_refs/` файлы:
`john.png` (фото Джона) и `mary.jpg` (фото Мэри).
Требования: одно лицо в кадре, резко, без очков и фильтров, лицо занимает 35–65% кадра.

In [ ]:
import os, shutil
REF_JOHN = f"{DIRS['refs']}/john.png"
REF_MARY = f"{DIRS['refs']}/mary.jpg"

missing = [p for p in (REF_JOHN, REF_MARY) if not os.path.exists(p)]
if missing:
    print('Нет файлов:', missing)
    print('Загрузите их вручную:')
    from google.colab import files
    up = files.upload()
    for name, data in up.items():
        target = REF_JOHN if 'john' in name.lower() else REF_MARY
        open(target, 'wb').write(data)
        print('сохранено ->', target)

for p in (REF_JOHN, REF_MARY):
    print(('OK  ' if os.path.exists(p) else 'НЕТ '), p)

# копия в input ComfyUI
os.makedirs('/content/ComfyUI/input', exist_ok=True)
for p, n in ((REF_JOHN,'john.png'), (REF_MARY,'mary.jpg')):
    if os.path.exists(p):
        shutil.copy(p, f'/content/ComfyUI/input/{n}')

## 6. Промпт-пак (59 планов)
Пак скачивается из GitHub при заданном `ASSET_BASE_URL` и кэшируется на Drive.
Файлы лиц никогда не скачиваются из публичного репозитория.

In [ ]:
import json, os, urllib.request
PACK = f'{PROJ}/shots_arcane.json'
if ASSET_BASE_URL:
    try:
        urllib.request.urlretrieve(f'{ASSET_BASE_URL}/shots_arcane.json', PACK)
        print('shots_arcane.json обновлён из GitHub')
    except Exception as e:
        print('GitHub raw пока недоступен:',e,'| использую Drive-кэш')
if not os.path.exists(PACK):
    from google.colab import files
    print('Загрузите shots_arcane.json'); up = files.upload()
    open(PACK,'wb').write(list(up.values())[0])

pack  = json.load(open(PACK, encoding='utf-8'))
SHOTS = pack['shots']
total = sum(s['duration_sec'] for s in SHOTS)
assert len(SHOTS) == 59, len(SHOTS)
assert abs(total - 311.5) < 0.01, total
assert all(s['face_order']==['mary','john'] for s in SHOTS if s['who']=='both')
assert all(not s['apply_face_id'] for s in SHOTS if s['non_explicit_water_scene'] and s['who']=='none')
print(f"{len(SHOTS)} планов | {total:.0f} c ({int(total//60)}:{int(total%60):02d})")
print('Политика:', pack['content_policy']['note'])
for s in SHOTS[:3]:
    print(f"\n{s['shot_id']} [{s['section']}] {s['duration_sec']}c who={s['who']}")
    print(' ', s['keyframe_prompt'][:160], '...')

## 7. Запуск сервера ComfyUI

In [ ]:
import subprocess, time, urllib.request, os, json
os.makedirs('/content/logs', exist_ok=True)
LOG = open('/content/logs/comfy.log','w')
try:
    old_proc = globals().get('proc')
    if old_proc and old_proc.poll() is None:
        old_proc.terminate(); old_proc.wait(timeout=15)
except Exception:
    pass
proc = subprocess.Popen(
    ['python','main.py','--listen','127.0.0.1','--port','8188','--lowvram','--preview-method','none'],
    cwd='/content/ComfyUI', stdout=LOG, stderr=subprocess.STDOUT)

for i in range(180):
    try:
        urllib.request.urlopen('http://127.0.0.1:8188/system_stats', timeout=2)
        print('ComfyUI поднялся за', i, 'c'); break
    except Exception:
        time.sleep(1)
else:
    raise RuntimeError(open('/content/logs/comfy.log').read()[-5000:])

required_nodes = {
 'CheckpointLoaderSimple','LoraLoader','CLIPTextEncode','KSampler','VAEDecode',
 'InstantIDModelLoader','InstantIDFaceAnalysis','ApplyInstantID','LoadImageMask',
 'SetLatentNoiseMask','UpscaleModelLoader','ImageUpscaleWithModel','ImageScale',
 'CLIPLoader','LTXVPreprocess','LTXVImgToVideo','LTXVConditioning',
 'LTXVScheduler','KSamplerSelect','SamplerCustom','VHS_VideoCombine'}
objects = json.loads(urllib.request.urlopen('http://127.0.0.1:8188/object_info').read())
missing_nodes = sorted(required_nodes - set(objects))
if missing_nodes:
    raise RuntimeError(f'Нет нод: {missing_nodes}\n' + open('/content/logs/comfy.log').read()[-5000:])
print('Схема нод проверена:', len(required_nodes), 'шт.')

## 8. Клиент ComfyUI API

In [ ]:
import json, urllib.request, uuid, time, os, glob
SRV = 'http://127.0.0.1:8188'
CID = str(uuid.uuid4())

def queue(workflow, timeout=900):
    """Отправляет workflow и ждёт результат. Возвращает список путей к файлам."""
    data = json.dumps({'prompt': workflow, 'client_id': CID}).encode()
    req  = urllib.request.Request(f'{SRV}/prompt', data=data,
                                  headers={'Content-Type':'application/json'})
    try:
        response = json.loads(urllib.request.urlopen(req).read())
    except urllib.error.HTTPError as exc:
        raise RuntimeError(exc.read().decode('utf-8','replace')) from exc
    if response.get('node_errors'):
        raise RuntimeError(json.dumps(response['node_errors'], ensure_ascii=False, indent=2))
    pid = response['prompt_id']
    t0 = time.time()
    while time.time() - t0 < timeout:
        h = json.loads(urllib.request.urlopen(f'{SRV}/history/{pid}').read())
        if pid in h:
            status = h[pid].get('status', {})
            if status.get('status_str') == 'error':
                raise RuntimeError(json.dumps(status, ensure_ascii=False, indent=2))
            out = []
            for node in h[pid].get('outputs', {}).values():
                for key in ('images','gifs','videos'):
                    for im in node.get(key, []):
                        sub = im.get('subfolder','')
                        out.append(os.path.join('/content/ComfyUI/output', sub, im['filename']))
            if not out:
                raise RuntimeError(f'Workflow {pid} завершился без файла')
            return out
        time.sleep(2)
    raise TimeoutError(f'Таймаут {timeout}c на {pid}')

print('Клиент готов')

## 9. Ключевые кадры (59 шт)
**ТОЧКА 1:** сначала генерируются только SH001–SH006. Посмотрите контактный лист,
оцените Arcane-стиль и палитру. Только после явного «ок» переключите
`GATE_1_APPROVED = True` и перезапустите эту ячейку для остальных кадров.

In [ ]:
ARCANE_LORA = 'arcane_style_xl_v1.safetensors'
ARCANE_WEIGHT = 0.80

def wf_keyframe(prompt, negative, seed, w=1024, h=576, steps=30, cfg=5.5):
    return {
      '1': {'class_type':'CheckpointLoaderSimple',
            'inputs':{'ckpt_name':'sd_xl_base_1.0.safetensors'}},
      '2': {'class_type':'LoraLoader','inputs':{
                'model':['1',0],'clip':['1',1],'lora_name':ARCANE_LORA,
                'strength_model':ARCANE_WEIGHT,'strength_clip':ARCANE_WEIGHT}},
      '3': {'class_type':'CLIPTextEncode','inputs':{'text':prompt,'clip':['2',1]}},
      '4': {'class_type':'CLIPTextEncode','inputs':{'text':negative,'clip':['2',1]}},
      '5': {'class_type':'EmptyLatentImage','inputs':{'width':w,'height':h,'batch_size':1}},
      '6': {'class_type':'KSampler','inputs':{
                'seed':seed,'steps':steps,'cfg':cfg,
                'sampler_name':'dpmpp_2m','scheduler':'karras','denoise':1.0,
                'model':['2',0],'positive':['3',0],'negative':['4',0],'latent_image':['5',0]}},
      '7': {'class_type':'VAEDecode','inputs':{'samples':['6',0],'vae':['1',2]}},
      '8': {'class_type':'SaveImage','inputs':{'filename_prefix':'kf','images':['7',0]}},
    }

import shutil
GATE_1_APPROVED = False  # менять на True только после одобрения контактного листа 6 кадров
KEYFRAME_SHOTS = SHOTS if GATE_1_APPROVED else SHOTS[:6]
done = skipped = 0
for s in KEYFRAME_SHOTS:
    dst = f"{DIRS['keyframes']}/{s['shot_id']}.png"
    if os.path.exists(dst):
        skipped += 1; continue
    try:
        files_out = queue(wf_keyframe(s['keyframe_prompt'], s['keyframe_negative'], s['seed']))
        shutil.copy(files_out[0], dst)
        done += 1
        print(f"[{done+skipped}/{len(KEYFRAME_SHOTS)}] {s['shot_id']} OK")
    except Exception as e:
        raise RuntimeError(f"{s['shot_id']} keyframe failed: {e}") from e

print(f'\nГотово: {done} новых, {skipped} уже было')
if not GATE_1_APPROVED:
    print('СТОП — ТОЧКА 1. Покажите _contact_sheet.jpg и ждите одобрения перед 59 кадрами.')

### 9b. Просмотр контактного листа

In [ ]:
import glob
from PIL import Image, ImageDraw
paths = [f"{DIRS['keyframes']}/{s['shot_id']}.png" for s in KEYFRAME_SHOTS
         if os.path.exists(f"{DIRS['keyframes']}/{s['shot_id']}.png")]
print(len(paths), 'кадров')
if paths:
    cols, tw = 6, 320
    rows = (len(paths)+cols-1)//cols
    th = int(tw*9/16)
    sheet = Image.new('RGB', (cols*tw, rows*th), (10,20,30))
    draw = ImageDraw.Draw(sheet)
    for i, p in enumerate(paths):
        x,y=(i%cols)*tw,(i//cols)*th
        sheet.paste(Image.open(p).convert('RGB').resize((tw,th)), (x,y))
        draw.rectangle((x,y,x+112,y+22),fill=(0,0,0))
        draw.text((x+5,y+4),os.path.basename(p)[:-4],fill=(255,255,255))
    sheet.save(f"{DIRS['keyframes']}/_contact_sheet.jpg", quality=88)
    display(sheet.resize((cols*tw//2, rows*th//2)))

# После просмотра внесите ID неудачных планов и перезапустите эту ячейку.
# Кадр и зависимый кэш удаляются, затем кадр строится с новым seed.
REGENERATE = []  # пример: ['S01_SH014','S01_SH029']
shot_by_id={s['shot_id']:s for s in SHOTS}
for sid in REGENERATE:
    if sid not in shot_by_id: raise KeyError(sid)
    s=shot_by_id[sid]; s['seed'] += 9000001
    for key in ('keyframes','faces','upscaled','clips'):
        ext='mp4' if key=='clips' else 'png'; p=f"{DIRS[key]}/{sid}.{ext}"
        if os.path.exists(p): os.unlink(p)
    out=queue(wf_keyframe(s['keyframe_prompt'],s['keyframe_negative'],s['seed']))
    shutil.copy(out[0],f"{DIRS['keyframes']}/{sid}.png")
    print('перегенерирован:',sid,'seed',s['seed'])

## 10. Подстановка лиц (InstantID)
38 планов с лицами. Для парных планов — **два прохода по очереди**
(сначала Мэри, потом Джон). Каждый проход имеет свою маску и target-keypoints;
промежуточные файлы сохраняются. **ТОЧКА 2:** сначала только SH003 (Мэри),
SH008 (Джон) и SH012 (пара). После одобрения выставьте `GATE_2_APPROVED = True`.

In [ ]:
import cv2, json, numpy as np
from insightface.app import FaceAnalysis

FACE_APP = FaceAnalysis(name='antelopev2', root='/content/ComfyUI/models/insightface',
                        providers=['CUDAExecutionProvider','CPUExecutionProvider'])
FACE_APP.prepare(ctx_id=0, det_size=(640,640))

def _area(face):
    x1,y1,x2,y2 = map(float, face.bbox); return max(0,x2-x1)*max(0,y2-y1)
def _embedding(face):
    v=np.asarray(face.embedding,dtype=np.float32); return v/max(np.linalg.norm(v),1e-8)
def detect_faces(path):
    image=cv2.imread(path)
    if image is None: raise RuntimeError(f'Cannot read image: {path}')
    return image, FACE_APP.get(image)

REF_EMB={}
for who,path in {'mary':REF_MARY,'john':REF_JOHN}.items():
    _,faces=detect_faces(path)
    if not faces: raise RuntimeError(f'No face in {who} reference: {path}')
    REF_EMB[who]=_embedding(max(faces,key=_area))

def target_assets(src, shot, who, prefix):
    image,faces=detect_faces(src); expected=2 if shot['who']=='both' else 1
    if len(faces)<expected:
        raise RuntimeError(f"{shot['shot_id']}: detected {len(faces)}/{expected} target faces; regenerate keyframe")
    if expected==2:
        faces=sorted(sorted(faces,key=_area,reverse=True)[:2],key=lambda f:float(f.bbox[0]))
        target=faces[0] if shot['face_targets'][who]=='left' else faces[1]
        centers=[(float(f.bbox[0])+float(f.bbox[2]))/2 for f in faces]
        split=int(round(sum(centers)/2))
    else:
        target=max(faces,key=_area); split=None
    h,w=image.shape[:2]; x1,y1,x2,y2=map(float,target.bbox); fw,fh=x2-x1,y2-y1
    xa=max(0,int(x1-.45*fw)); xb=min(w,int(x2+.45*fw))
    ya=max(0,int(y1-.75*fh)); yb=min(h,int(y2+.45*fh))
    mask=np.zeros((h,w),np.uint8)
    cv2.ellipse(mask,((xa+xb)//2,(ya+yb)//2),(max(1,(xb-xa)//2),max(1,(yb-ya)//2)),0,0,360,255,-1)
    if split is not None:
        if who=='mary': mask[:,split:]=0
        else: mask[:,:split]=0
    mask=cv2.GaussianBlur(mask,(0,0),sigmaX=max(4,fw*.08),sigmaY=max(4,fh*.08))
    kps=np.zeros_like(image); kps[ya:yb,xa:xb]=image[ya:yb,xa:xb]
    mask_name=f'{prefix}_mask.png'; kps_name=f'{prefix}_kps.png'
    cv2.imwrite(f'{IN}/{mask_name}',mask); cv2.imwrite(f'{IN}/{kps_name}',kps)
    return mask_name,kps_name

def wf_face(image_name, ref_name, mask_name, kps_name, prompt, negative, seed,
            weight=0.85, denoise=0.38):
    """Один маскированный InstantID-проход ровно для одной личности."""
    return {
      '1': {'class_type':'CheckpointLoaderSimple',
            'inputs':{'ckpt_name':'sd_xl_base_1.0.safetensors'}},
      '2': {'class_type':'LoraLoader','inputs':{
                'model':['1',0],'clip':['1',1],'lora_name':ARCANE_LORA,
                'strength_model':ARCANE_WEIGHT,'strength_clip':ARCANE_WEIGHT}},
      '3': {'class_type':'LoadImage','inputs':{'image':image_name}},
      '4': {'class_type':'LoadImage','inputs':{'image':ref_name}},
      '5': {'class_type':'LoadImage','inputs':{'image':kps_name}},
      '6': {'class_type':'LoadImageMask','inputs':{'image':mask_name,'channel':'red'}},
      '7': {'class_type':'InstantIDModelLoader','inputs':{'instantid_file':'ip-adapter.bin'}},
      '8': {'class_type':'InstantIDFaceAnalysis','inputs':{'provider':'CUDA'}},
      '9': {'class_type':'ControlNetLoader',
            'inputs':{'control_net_name':'instantid_controlnet.safetensors'}},
      '10':{'class_type':'CLIPTextEncode','inputs':{'text':prompt,'clip':['2',1]}},
      '11':{'class_type':'CLIPTextEncode','inputs':{'text':negative,'clip':['2',1]}},
      '12':{'class_type':'ApplyInstantID','inputs':{
                'instantid':['7',0],'insightface':['8',0],'control_net':['9',0],
                'image':['4',0],'image_kps':['5',0],'mask':['6',0],
                'model':['2',0],'positive':['10',0],'negative':['11',0],
                'weight':weight,'start_at':0.0,'end_at':0.80}},
      '13':{'class_type':'VAEEncode','inputs':{'pixels':['3',0],'vae':['1',2]}},
      '14':{'class_type':'SetLatentNoiseMask','inputs':{'samples':['13',0],'mask':['6',0]}},
      '15':{'class_type':'KSampler','inputs':{
                'seed':seed,'steps':28,'cfg':4.5,
                'sampler_name':'dpmpp_2m','scheduler':'karras','denoise':denoise,
                'model':['12',0],'positive':['12',1],'negative':['12',2],'latent_image':['14',0]}},
      '16':{'class_type':'VAEDecode','inputs':{'samples':['15',0],'vae':['1',2]}},
      '17':{'class_type':'SaveImage','inputs':{'filename_prefix':'face','images':['16',0]}},
    }

REFS = {'john':'john.png', 'mary':'mary.jpg'}
IN   = '/content/ComfyUI/input'
PASS_DIR = f"{DIRS['faces']}/_passes"; os.makedirs(PASS_DIR, exist_ok=True)
if not GATE_1_APPROVED: raise RuntimeError('Сначала получите одобрение ТОЧКИ 1 и завершите 59 keyframes')
FACE_SMOKE_IDS = ['S01_SH003','S01_SH008','S01_SH012']
GATE_2_APPROVED = False  # True только после одобрения трёх контрольных лиц
FACE_SHOTS = SHOTS if GATE_2_APPROVED else [s for s in SHOTS if s['shot_id'] in FACE_SMOKE_IDS]
done = skipped = 0

for s in FACE_SHOTS:
    dst = f"{DIRS['faces']}/{s['shot_id']}.png"
    src = f"{DIRS['keyframes']}/{s['shot_id']}.png"
    if os.path.exists(dst): skipped += 1; continue
    if not os.path.exists(src): raise FileNotFoundError(src)
    if not s['apply_face_id']:
        shutil.copy(src, dst); skipped += 1; continue
    try:
        cur = src
        for pass_no, who in enumerate(s['face_order'], 1): # строго mary -> john
            prefix = f"{s['shot_id']}_{pass_no}_{who}"
            tmp = f'{IN}/{prefix}_input.png'
            shutil.copy(cur, tmp)
            mask_name, kps_name = target_assets(tmp, s, who, prefix)
            out = queue(wf_face(os.path.basename(tmp), REFS[who], mask_name, kps_name,
                                s['keyframe_prompt'], s['keyframe_negative'],
                                s['seed'] + pass_no*100003))
            cur = out[0]
            shutil.copy(cur, f'{PASS_DIR}/{prefix}.png')
        shutil.copy(cur, dst)
        _, detected = detect_faces(dst)
        if len(detected) < len(s['face_order']):
            raise RuntimeError(f"{s['shot_id']}: output face detection {len(detected)}/{len(s['face_order'])}")
        done += 1
        print(f"[{done+skipped}/{len(FACE_SHOTS)}] {s['shot_id']} лица: {s['face_order']}")
    except Exception as e:
        raise RuntimeError(f"{s['shot_id']} InstantID failed; no fallback allowed: {e}") from e

# QC проверяет активную партию; после gate 2 — все 38 face-планов.
face_qc=[]
for s in FACE_SHOTS:
    if not s['apply_face_id']: continue
    p=f"{DIRS['faces']}/{s['shot_id']}.png"; _,detected=detect_faces(p)
    expected=len(s['face_order'])
    if len(detected)<expected: raise RuntimeError(f"{s['shot_id']}: QC faces {len(detected)}/{expected}")
    chosen=(sorted(sorted(detected,key=_area,reverse=True)[:2],key=lambda f:float(f.bbox[0]))
            if s['who']=='both' else [max(detected,key=_area)])
    roles=['mary','john'] if s['who']=='both' else s['face_order']
    for role,face in zip(roles,chosen):
        emb=_embedding(face); own=float(emb@REF_EMB[role])
        other=float(emb@REF_EMB['john' if role=='mary' else 'mary'])
        face_qc.append({'shot_id':s['shot_id'],'role':role,'own_similarity':round(own,4),
                        'other_similarity':round(other,4),'identity_margin':round(own-other,4)})
qc_path=f"{DIRS['faces']}/face_identity_qc.json"
json.dump(face_qc,open(qc_path,'w'),ensure_ascii=False,indent=2)
mixed=[r for r in face_qc if r['identity_margin'] <= 0]
if mixed: raise RuntimeError(f'Возможное смешение личностей: {mixed[:8]}')
smoke_paths=[f"{DIRS['faces']}/{sid}.png" for sid in FACE_SMOKE_IDS]
if all(os.path.exists(p) for p in smoke_paths):
    face_sheet=Image.new('RGB',(960,180),(10,20,30))
    fd=ImageDraw.Draw(face_sheet)
    for i,p in enumerate(smoke_paths):
        face_sheet.paste(Image.open(p).convert('RGB').resize((320,180)),(i*320,0))
        fd.rectangle((i*320,0,i*320+112,22),fill=(0,0,0)); fd.text((i*320+5,4),FACE_SMOKE_IDS[i],fill='white')
    face_sheet.save(f"{DIRS['faces']}/_checkpoint_2_faces.jpg",quality=90); display(face_sheet)
print(f'\nГотово: {done} новых, {skipped} пропущено')
if not GATE_2_APPROVED:
    print('СТОП — ТОЧКА 2. Покажите _checkpoint_2_faces.jpg и ждите одобрения.')

## 11. Нейронный апскейл до 1080p (4x-UltraSharp)

In [ ]:
if not GATE_2_APPROVED: raise RuntimeError('Сначала получите одобрение ТОЧКИ 2 и завершите face-проходы')
def wf_upscale(image_name):
    return {
      '1':{'class_type':'LoadImage','inputs':{'image':image_name}},
      '2':{'class_type':'UpscaleModelLoader','inputs':{'model_name':'4x-UltraSharp.pth'}},
      '3':{'class_type':'ImageUpscaleWithModel','inputs':{'upscale_model':['2',0],'image':['1',0]}},
      '4':{'class_type':'ImageScale','inputs':{'image':['3',0],'upscale_method':'lanczos',
            'width':1920,'height':1080,'crop':'disabled'}},
      '5':{'class_type':'SaveImage','inputs':{'filename_prefix':'upscaled','images':['4',0]}},
    }

n = 0
for s in SHOTS:
    src = f"{DIRS['faces']}/{s['shot_id']}.png"
    dst = f"{DIRS['upscaled']}/{s['shot_id']}.png"
    if os.path.exists(dst): continue
    if not os.path.exists(src): raise FileNotFoundError(src)
    name=f"up_{s['shot_id']}.png"; shutil.copy(src,f'{IN}/{name}')
    try:
        out=queue(wf_upscale(name),timeout=900); shutil.copy(out[0],dst)
    except Exception as e:
        raise RuntimeError(f"{s['shot_id']} upscale failed: {e}") from e
    n += 1
bad=[]
for s in SHOTS:
    p=f"{DIRS['upscaled']}/{s['shot_id']}.png"; im=cv2.imread(p)
    if im is None or im.shape[1]!=1920 or im.shape[0]!=1080: bad.append(s['shot_id'])
if bad: raise RuntimeError(f'Bad upscale outputs: {bad}')
print(n, 'новых кадров; 59/59 проверены как 1920x1080')

## 12. Анимация кадров (image-to-video)
Каждый кадр оживает в короткий клип по своему motion-промпту.
**ТОЧКА 3:** сначала генерируются два face-клипа SH003 и SH008. Проверьте
стабильность лиц и движение; после одобрения выставьте `GATE_3_APPROVED = True`.

Официальная цепочка LTX 0.9.5: отдельный T5XXL, preprocess, LTX-conditioning,
LTX-scheduler и custom sampler. 97 кадров дают ~4 с настоящего движения.

In [ ]:
def wf_i2v(image_name, prompt, negative, frames=97, w=768, h=448, seed=0):
    """LTX-Video 0.9.5 image-to-video, official ComfyUI node chain."""
    return {
      '1': {'class_type':'CheckpointLoaderSimple',
            'inputs':{'ckpt_name':'ltx-video-2b-v0.9.5.safetensors'}},
      '2': {'class_type':'CLIPLoader','inputs':{
            'clip_name':'t5xxl_fp16.safetensors','type':'ltxv','device':'default'}},
      '3': {'class_type':'LoadImage','inputs':{'image':image_name}},
      '4': {'class_type':'LTXVPreprocess','inputs':{'image':['3',0],'img_compression':40}},
      '5': {'class_type':'CLIPTextEncode','inputs':{'text':prompt,'clip':['2',0]}},
      '6': {'class_type':'CLIPTextEncode','inputs':{'text':negative,'clip':['2',0]}},
      '7': {'class_type':'LTXVImgToVideo','inputs':{
                'positive':['5',0],'negative':['6',0],'vae':['1',2],'image':['4',0],
                'width':w,'height':h,'length':frames,'batch_size':1,'strength':1.0}},
      '8': {'class_type':'LTXVConditioning','inputs':{
                'positive':['7',0],'negative':['7',1],'frame_rate':24.0}},
      '9': {'class_type':'LTXVScheduler','inputs':{
                'steps':30,'max_shift':2.05,'base_shift':0.95,'stretch':True,
                'terminal':0.1,'latent':['7',2]}},
      '10':{'class_type':'KSamplerSelect','inputs':{'sampler_name':'euler'}},
      '11':{'class_type':'SamplerCustom','inputs':{
                'model':['1',0],'add_noise':True,'noise_seed':seed,'cfg':3.0,
                'positive':['8',0],'negative':['8',1],'sampler':['10',0],
                'sigmas':['9',0],'latent_image':['7',2]}},
      '12':{'class_type':'VAEDecode','inputs':{'samples':['11',0],'vae':['1',2]}},
      '13':{'class_type':'VHS_VideoCombine','inputs':{
                'images':['12',0],'frame_rate':24,'loop_count':0,
                'filename_prefix':'clip','format':'video/h264-mp4',
                'pix_fmt':'yuv420p','crf':19,'save_metadata':True,'trim_to_audio':False,
                'pingpong':False,'save_output':True}},
    }

MAX_FRAMES = 97          # LTX требует 8n+1; ~4.0 c при 24 fps
if not GATE_2_APPROVED: raise RuntimeError('Сначала получите одобрение ТОЧКИ 2')
I2V_SMOKE_IDS = ['S01_SH003','S01_SH008']
GATE_3_APPROVED = False  # True только после просмотра двух контрольных клипов
VIDEO_SHOTS = SHOTS if GATE_3_APPROVED else [s for s in SHOTS if s['shot_id'] in I2V_SMOKE_IDS]
done = skipped = 0
for s in VIDEO_SHOTS:
    dst = f"{DIRS['clips']}/{s['shot_id']}.mp4"
    src = f"{DIRS['upscaled']}/{s['shot_id']}.png"
    if os.path.exists(dst): skipped += 1; continue
    if not os.path.exists(src): raise FileNotFoundError(src)
    try:
        name = f"{s['shot_id']}.png"
        shutil.copy(src, f'{IN}/{name}')
        frames = min(MAX_FRAMES, s['frames_24fps'])
        out = queue(wf_i2v(name, s['motion_prompt'], s['motion_negative'],
                           frames=frames, seed=s['seed']), timeout=1800)
        shutil.copy(out[0], dst)
        done += 1
        print(f"[{done+skipped}/{len(VIDEO_SHOTS)}] {s['shot_id']} {frames}к OK")
    except Exception as e:
        raise RuntimeError(f"{s['shot_id']} i2v failed: {e}") from e

clips=[f"{DIRS['clips']}/{s['shot_id']}.mp4" for s in VIDEO_SHOTS]
missing=[p for p in clips if not os.path.exists(p)]
if missing: raise RuntimeError(f'Missing i2v clips: {missing[:8]}')

# Face-motion QC: начало/середина/конец каждого face-клипа.
motion_qc=[]; motion_bad=[]
motion_dir=f"{DIRS['clips']}/_face_qc"; os.makedirs(motion_dir,exist_ok=True)
for s in VIDEO_SHOTS:
    if not s['apply_face_id']: continue
    cap=cv2.VideoCapture(f"{DIRS['clips']}/{s['shot_id']}.mp4")
    count=int(cap.get(cv2.CAP_PROP_FRAME_COUNT)); baseline={}
    for frac in (0.05,0.50,0.95):
        idx=max(0,min(count-1,int((count-1)*frac))); cap.set(cv2.CAP_PROP_POS_FRAMES,idx)
        ok,frame=cap.read(); frame_path=f"{motion_dir}/{s['shot_id']}_{idx:04d}.jpg"
        if not ok: motion_bad.append({'shot_id':s['shot_id'],'frame':idx,'reason':'decode'}); continue
        cv2.imwrite(frame_path,frame); faces=FACE_APP.get(frame)
        expected=len(s['face_order'])
        if len(faces)<expected:
            motion_bad.append({'shot_id':s['shot_id'],'frame':idx,'reason':f'faces {len(faces)}/{expected}'})
            continue
        chosen=(sorted(sorted(faces,key=_area,reverse=True)[:2],key=lambda f:float(f.bbox[0]))
                if s['who']=='both' else [max(faces,key=_area)])
        roles=['mary','john'] if s['who']=='both' else s['face_order']
        for role,face in zip(roles,chosen):
            emb=_embedding(face); own=float(emb@REF_EMB[role])
            other=float(emb@REF_EMB['john' if role=='mary' else 'mary']); margin=own-other
            baseline.setdefault(role,own); drift=baseline[role]-own
            rec={'shot_id':s['shot_id'],'frame':idx,'role':role,'own_similarity':round(own,4),
                 'identity_margin':round(margin,4),'similarity_drop':round(drift,4)}
            motion_qc.append(rec)
            if margin<=0 or drift>0.15: motion_bad.append({**rec,'reason':'identity drift'})
    cap.release()
json.dump({'samples':motion_qc,'failed':motion_bad},
          open(f"{DIRS['clips']}/face_motion_qc.json",'w'),ensure_ascii=False,indent=2)
if motion_bad: raise RuntimeError(f'Лица плывут/не распознаны; перерендерить seed: {motion_bad[:8]}')
from IPython.display import Video, display
if not GATE_3_APPROVED:
    for sid in I2V_SMOKE_IDS: display(Video(f"{DIRS['clips']}/{sid}.mp4",embed=True,width=640))
    print('СТОП — ТОЧКА 3. Покажите два клипа и ждите одобрения.')
else:
    print(f'\nКлипов: {done} новых, {skipped} из кэша; 59/59 на месте; face-motion QC пройден')

## 13. Сборка мастера
Каждый клип растягивается до своей длительности по раскадровке,
приводится к 1920×1080 / 24 fps и склеивается. Статичный fallback запрещён:
при отсутствии хотя бы одного i2v-клипа сборка останавливается.

In [ ]:
import subprocess, os
if not GATE_3_APPROVED: raise RuntimeError('Сначала получите одобрение ТОЧКИ 3 и завершите 59 i2v-клипов')
WORK = '/content/assembly'; os.makedirs(WORK, exist_ok=True)

segments = []
for s in SHOTS:
    clip = f"{DIRS['clips']}/{s['shot_id']}.mp4"
    seg = f"{WORK}/{s['shot_id']}.mp4"
    dur = s['duration_sec']
    if not os.path.exists(clip): raise FileNotFoundError(f'Нет i2v-клипа: {clip}')
    probe = subprocess.run(['ffprobe','-v','error','-show_entries','format=duration',
                            '-of','csv=p=0', clip], capture_output=True, text=True, check=True)
    have=float(probe.stdout.strip())
    if have<=0: raise RuntimeError(f'Нулевая длительность: {clip}')
    speed=have/dur
    vf=f"setpts={1/speed:.10f}*PTS,scale=1920:1080:force_original_aspect_ratio=increase,crop=1920:1080,fps=24"
    cmd=['ffmpeg','-y','-i',clip,'-vf',vf,'-t',str(dur),
         '-an','-c:v','libx264','-profile:v','high','-pix_fmt','yuv420p','-crf','18',seg]
    subprocess.run(cmd,capture_output=True,text=True,check=True)
    segments.append(seg)

if len(segments)!=59: raise RuntimeError(f'Сегментов {len(segments)}/59')
print(len(segments), 'сегментов готово')
lst = f'{WORK}/list.txt'
open(lst,'w').write('\n'.join(f"file '{p}'" for p in segments))

MASTER = f"{DIRS['delivery']}/PLYAZH_V2_CLEAN_1080P24.mp4"
subprocess.run(['ffmpeg','-y','-f','concat','-safe','0','-i',lst,
                '-c:v','libx264','-profile:v','high','-pix_fmt','yuv420p',
                '-colorspace','bt709','-crf','18','-r','24', MASTER],
               capture_output=True,text=True,check=True)

d = subprocess.run(['ffprobe','-v','error','-show_entries','format=duration',
                    '-of','csv=p=0', MASTER], capture_output=True, text=True).stdout.strip()
if float(d)<311.4: raise RuntimeError(f'Мастер короче брифа: {d} c')
print('МАСТЕР:', MASTER)
print('Длительность:', d, 'c')

## 14. Озвучка — красивые русские голоса

Четыре роли и три базовых TTS-тембра вместо системного TTS. Мысли Мэри
звучат её же голосом; контраст создают темп, высота, громкость и реверберация:

| Роль | Голос | Характер |
|---|---|---|
| Рассказчик | `ru-RU-DmitryNeural` −12% темпа | спокойный, тёплый, с паузами |
| Джон | `en-US-AndrewMultilingualNeural` | мягкий уверенный мужской |
| Мэри | `ru-RU-SvetlanaNeural` | живая, играющая |
| Мысли Мэри | `ru-RU-SvetlanaNeural` −14%, ниже и тише | тот же человек + реверберация |

**Edge TTS** — нейронные голоса Microsoft: бесплатно, без ключа и регистрации,
качество несравнимо выше `macOS say`. Ниже есть альтернативы: Silero (офлайн)
и XTTS v2 (клонирование — можно озвучить героев собственными голосами).

In [ ]:
!pip -q install edge-tts
import json, os

VPACK = f'{PROJ}/voice_pack.json'
if ASSET_BASE_URL:
    try:
        urllib.request.urlretrieve(f'{ASSET_BASE_URL}/voice_pack.json', VPACK)
        print('voice_pack.json обновлён из GitHub')
    except Exception as e:
        print('GitHub raw пока недоступен:',e,'| использую Drive-кэш')
if not os.path.exists(VPACK):
    from google.colab import files
    print('Загрузите voice_pack.json'); up = files.upload()
    open(VPACK,'wb').write(list(up.values())[0])

vp     = json.load(open(VPACK, encoding='utf-8'))
VOICES = vp['voices']
LINES  = vp['lines']
assert len(LINES)==39, len(LINES)
assert len(VOICES)==4, 'Нужны четыре роли'
assert len({v['edge']['voice'] for v in VOICES.values()})==3, 'Ожидаются три базовых тембра'
assert VOICES['mary']['edge']['voice']==VOICES['mary_inner']['edge']['voice']
print(f"{len(LINES)} реплик | фильм {vp['film_duration_sec']} c")
for k, n in vp['speaker_line_counts'].items():
    v = VOICES[k]['edge']
    print(f"  {VOICES[k]['title']:24s} {n:2d} реплик  {v['voice']} ({v['rate']}, {v['pitch']})")

# посмотреть все доступные русские голоса
!edge-tts --list-voices | grep -i 'ru-RU' || true

### 14b. Синтез реплик

In [ ]:
import asyncio, edge_tts, os
VO = f"{DIRS['audio']}/lines"; os.makedirs(VO, exist_ok=True)

async def say(line):
    cfg = VOICES[line['speaker']]['edge']
    dst = f"{VO}/{line['id']}_{line['speaker']}.mp3"
    if os.path.exists(dst) and os.path.getsize(dst) > 2000:
        return dst, True
    com = edge_tts.Communicate(line['text'], cfg['voice'],
                               rate=cfg['rate'], pitch=cfg['pitch'],
                               volume=cfg.get('volume','+0%'))
    await com.save(dst)
    return dst, False

async def run_all():
    new = skip = 0; errors=[]
    for l in LINES:
        try:
            _, cached = await say(l)
            skip += cached; new += (not cached)
        except Exception as e:
            print(f"[!] {l['id']} ({l['speaker']}): {e}")
            fb = VOICES[l['speaker']].get('edge_fallback')
            if fb:
                print('    пробую запасной голос', fb['voice'])
                c = edge_tts.Communicate(l['text'], fb['voice'],
                                         rate=fb.get('rate','+0%'), pitch=fb.get('pitch','+0Hz'))
                await c.save(f"{VO}/{l['id']}_{l['speaker']}.mp3")
                new += 1
            else:
                errors.append((l['id'],str(e)))
    if errors: raise RuntimeError(f'TTS errors: {errors}')
    print(f'Синтез: {new} новых, {skip} из кэша')

await run_all()
made=[f"{VO}/{l['id']}_{l['speaker']}.mp3" for l in LINES]
bad=[p for p in made if not os.path.exists(p) or os.path.getsize(p)<=2000]
if bad: raise RuntimeError(f'Не готовы реплики: {bad}')
print('39/39 файлов в', VO)

### 14c. Обработка и сборка голосовой дорожки
Каждой роли — своя окраска: рассказчик мягче и теплее, реплики ближе и чётче,
мысли Мэри — с лёгкой реверберацией. Затем всё раскладывается по таймкодам.

In [ ]:
import subprocess, os, json

# фильтры обработки под каждую роль
POST = {
  'eq_warm':        'highpass=f=80,equalizer=f=180:t=q:w=1:g=2,equalizer=f=3200:t=q:w=2:g=1.5,acompressor=threshold=-18dB:ratio=3:attack=8:release=180',
  'eq_male_close':  'highpass=f=90,equalizer=f=2800:t=q:w=2:g=2.5,acompressor=threshold=-16dB:ratio=3.5:attack=6:release=150',
  'eq_female_close':'highpass=f=110,equalizer=f=3400:t=q:w=2:g=2.5,acompressor=threshold=-16dB:ratio=3.5:attack=6:release=150',
  'intimate_reverb':'highpass=f=120,equalizer=f=3000:t=q:w=2:g=1.5,aecho=0.8:0.85:38:0.22,volume=-2dB',
}

PROC = f"{DIRS['audio']}/processed"; os.makedirs(PROC, exist_ok=True)
def tempo_filter(factor):
    parts=[]
    while factor>2.0: parts.append('atempo=2.0'); factor/=2.0
    while factor<0.5: parts.append('atempo=0.5'); factor/=0.5
    parts.append(f'atempo={factor:.6f}')
    return ','.join(parts)

delays, inputs = [], []
n = 0
for l in LINES:
    src = f"{VO}/{l['id']}_{l['speaker']}.mp3"
    if not os.path.exists(src): raise FileNotFoundError(src)
    dst = f"{PROC}/{l['id']}.wav"
    have=float(subprocess.run(['ffprobe','-v','error','-show_entries','format=duration','-of','csv=p=0',src],
                              capture_output=True,text=True,check=True).stdout)
    target=float(l['est_duration_sec']); speed=have/target
    af=POST.get(VOICES[l['speaker']]['post'],'highpass=f=80')+','+tempo_filter(speed)
    subprocess.run(['ffmpeg','-y','-i',src,'-af',af,'-t',str(target),'-ar','48000','-ac','1',dst],
                   capture_output=True,text=True,check=True)
    if os.path.exists(dst):
        inputs += ['-i', dst]
        delays.append((n, int(l['start_sec']*1000)))
        n += 1

if n!=39: raise RuntimeError(f'Обработано {n}/39 реплик')
print(n, 'реплик обработано')

# раскладываем по таймкодам и суммируем в одну дорожку
parts = [f'[{i}:a]adelay={ms}|{ms}[d{i}]' for i, ms in delays]
tags  = ''.join(f'[d{i}]' for i, _ in delays)
fc = (';'.join(parts) + f';{tags}amix=inputs={n}:duration=longest:normalize=0[vo0]'
      + f';[vo0]apad=whole_dur={vp["film_duration_sec"]},atrim=duration={vp["film_duration_sec"]}[vo]')
VOICE_TRACK = f"{DIRS['audio']}/voice.wav"
subprocess.run(['ffmpeg','-y'] + inputs +
               ['-filter_complex',fc,'-map','[vo]','-ar','48000','-ac','2', VOICE_TRACK],
               capture_output=True,text=True,check=True)

d = subprocess.run(['ffprobe','-v','error','-show_entries','format=duration','-of','csv=p=0',
                    VOICE_TRACK], capture_output=True, text=True).stdout.strip()
if float(d)<vp['film_duration_sec']-0.05: raise RuntimeError(f'voice.wav короткий: {d}')
print('Голосовая дорожка:', VOICE_TRACK, '|', d, 'c')
from IPython.display import Audio, display
display(Audio(VOICE_TRACK))

### 14d. Музыка MusicGen и морской эмбиент
После i2v ComfyUI освобождает GPU. В отдельном Python 3.10-окружении ставится
официальный `audiocraft`: MusicGen создаёт шесть музыкальных фрагментов, AudioGen —
четыре варианта спокойного ночного моря. Фрагменты склеиваются с кроссфейдами
и приводятся ровно к 311.5 секунды как `05_audio/music.wav` и `sea.wav`.

In [ ]:
import gc, glob, json, os, pathlib, subprocess, sys
A=DIRS['audio']; raw=f'{A}/audiocraft_raw'; os.makedirs(raw,exist_ok=True)
music_out,sea_out=f'{A}/music.wav',f'{A}/sea.wav'

# Визуальный этап завершён — освобождаем T4 для AudioCraft.
if proc.poll() is None:
    proc.terminate(); proc.wait(timeout=30)
gc.collect(); torch.cuda.empty_cache()
subprocess.run([sys.executable,'-m','pip','install','-q','uv'],check=True)
env='/content/audiocraft-env'
if not os.path.exists(f'{env}/bin/python'):
    subprocess.run(['uv','venv','--python','3.10',env],check=True)
py=f'{env}/bin/python'
marker=f'{env}/.plyazh-ready'
if not os.path.exists(marker):
    subprocess.run(['uv','pip','install','--python',py,'numpy<2','torch==2.1.0',
                    'torchaudio==2.1.0','audiocraft==1.3.0','soundfile'],check=True)
    pathlib.Path(marker).write_text('audiocraft 1.3.0 / torch 2.1.0')

audio_script=r'''
import gc, os, sys, torch
from audiocraft.models import MusicGen, AudioGen
from audiocraft.data.audio import audio_write
out=sys.argv[1]; os.makedirs(out,exist_ok=True)
base='slow cinematic romantic score, warm analog synth pads, soft piano, distant ocean, nostalgic, dreamy, 70 bpm, emotional, no drums'
music_prompts=[base+', '+v for v in [
 'delicate opening, sparse piano','quiet anticipation, warm unresolved harmony',
 'gentle romantic lift, luminous pads','intimate moonlit water, suspended chords',
 'tender emotional peak, restrained','soft reflective ending, slowly fading']]
missing=[i for i in range(len(music_prompts)) if not os.path.exists(f'{out}/music_{i:02d}.wav')]
if missing:
    model=MusicGen.get_pretrained('facebook/musicgen-small')
    model.set_generation_params(duration=30,use_sampling=True,top_k=250,temperature=1.0,cfg_coef=3.0)
    for i in missing:
        wav=model.generate([music_prompts[i]],progress=True)[0].cpu()
        audio_write(f'{out}/music_{i:02d}',wav,model.sample_rate,strategy='loudness',loudness_compressor=True)
    del model; gc.collect(); torch.cuda.empty_cache()
sea_prompts=[
 'calm night ocean waves rolling onto a sandy beach, distant gentle surf, no people, no music',
 'soft Mediterranean sea waves at dusk, light foam on wet sand, peaceful natural ambience',
 'quiet dark ocean shoreline, slow small waves, subtle sea breeze, clean field recording',
 'gentle moonlit surf washing over sand, serene continuous ocean ambience, no birds, no voices']
missing=[i for i in range(len(sea_prompts)) if not os.path.exists(f'{out}/sea_{i:02d}.wav')]
if missing:
    model=AudioGen.get_pretrained('facebook/audiogen-medium')
    model.set_generation_params(duration=10,use_sampling=True,top_k=250,temperature=1.0,cfg_coef=3.0)
    for i in missing:
        wav=model.generate([sea_prompts[i]],progress=True)[0].cpu()
        audio_write(f'{out}/sea_{i:02d}',wav,model.sample_rate,strategy='loudness',loudness_compressor=True)
'''
subprocess.run([py,'-c',audio_script,raw],check=True)

def crossfade_join(paths,out,fade):
    if len(paths)<2: raise RuntimeError(paths)
    args=sum((['-i',p] for p in paths),[])
    filters=[f'[{i}:a]aresample=48000,aformat=sample_fmts=fltp:channel_layouts=stereo[a{i}]' for i in range(len(paths))]
    last='a0'
    for i in range(1,len(paths)):
        tag=f'x{i}'; filters.append(f'[{last}][a{i}]acrossfade=d={fade}:c1=tri:c2=tri[{tag}]'); last=tag
    subprocess.run(['ffmpeg','-y']+args+['-filter_complex',';'.join(filters),'-map',f'[{last}]',
                    '-ar','48000','-ac','2','-c:a','pcm_s16le',out],capture_output=True,text=True,check=True)

music_raw=sorted(glob.glob(f'{raw}/music_*.wav')); sea_raw=sorted(glob.glob(f'{raw}/sea_*.wav'))
if len(music_raw)!=6 or len(sea_raw)!=4: raise RuntimeError((len(music_raw),len(sea_raw)))
music_join=f'{raw}/music_join.wav'; sea_join=f'{raw}/sea_join.wav'
crossfade_join(music_raw,music_join,3); crossfade_join(sea_raw,sea_join,2)
duration=str(vp['film_duration_sec'])
subprocess.run(['ffmpeg','-y','-stream_loop','-1','-i',music_join,'-t',duration,
                '-af','afade=t=in:st=0:d=2,afade=t=out:st=305.5:d=6',
                '-ar','48000','-ac','2','-c:a','pcm_s16le',music_out],capture_output=True,text=True,check=True)
subprocess.run(['ffmpeg','-y','-stream_loop','-1','-i',sea_join,'-t',duration,
                '-af','highpass=f=35,lowpass=f=9000,afade=t=in:st=0:d=2,afade=t=out:st=307.5:d=4',
                '-ar','48000','-ac','2','-c:a','pcm_s16le',sea_out],capture_output=True,text=True,check=True)
for p in (music_out,sea_out):
    d=float(subprocess.run(['ffprobe','-v','error','-show_entries','format=duration','-of','csv=p=0',p],
                           capture_output=True,text=True,check=True).stdout)
    if d<311.45: raise RuntimeError(f'{p}: {d}s')
    print(os.path.basename(p),f'{d:.2f}s',f'{os.path.getsize(p)/1024**2:.1f} MB')

### 14e. Субтитры из голосового пака
Тайминги уже посчитаны — SRT собирается автоматически.

In [ ]:
def ts(sec):
    h, rem = divmod(sec, 3600); m, s = divmod(rem, 60)
    return f'{int(h):02d}:{int(m):02d}:{int(s):02d},{int((s%1)*1000):03d}'

SRT = f"{DIRS['delivery']}/PLYAZH_V2_SUBTITLES_RU.srt"
with open(SRT,'w',encoding='utf-8') as f:
    for i, l in enumerate(LINES, 1):
        end = l['start_sec'] + l['est_duration_sec']
        who = '' if l['speaker']=='narrator' else f"{VOICES[l['speaker']]['title']}: "
        f.write(f"{i}\n{ts(l['start_sec'])} --> {ts(end)}\n{who}{l['text']}\n\n")
print('Субтитры:', SRT)
print(open(SRT, encoding='utf-8').read()[:400])

### 14f. Финальный микс
Голос 0 dB + музыка −11 dB + море −13 dB. Затем обязательная двухпроходная
нормализация EBU R128 к −14 LUFS / true peak −1 dBFS и проверка полной длины.

In [ ]:
import json, os, re, subprocess
A = DIRS['audio']
voice, music, sea = f'{A}/voice.wav', f'{A}/music.wav', f'{A}/sea.wav'
tracks = [(voice, 0.0), (music, -11.0), (sea, -13.0)]
missing=[p for p,_ in tracks if not os.path.exists(p)]
if missing: raise FileNotFoundError(f'Нет обязательных дорожек: {missing}')
duration=vp['film_duration_sec']
MIX_PRE=f'{A}/mix_pre.wav'; MIX_MASTER=f'{A}/mix_-14LUFS.wav'
inputs=sum((['-i',p] for p,_ in tracks),[])
filters=[f'[{i}:a]volume={db}dB,apad=whole_dur={duration},atrim=duration={duration}[a{i}]'
         for i,(_,db) in enumerate(tracks)]
tags=''.join(f'[a{i}]' for i in range(len(tracks)))
fc=';'.join(filters)+f';{tags}amix=inputs=3:duration=longest:normalize=0[mix]'
subprocess.run(['ffmpeg','-y']+inputs+['-filter_complex',fc,'-map','[mix]',
                '-ar','48000','-ac','2','-c:a','pcm_s24le',MIX_PRE],
               capture_output=True,text=True,check=True)
measure=subprocess.run(['ffmpeg','-hide_banner','-i',MIX_PRE,'-af',
                        'loudnorm=I=-14:TP=-1:LRA=11:print_format=json',
                        '-f','null','-'],capture_output=True,text=True,check=True)
blocks=re.findall(r'\{[\s\S]*?\}',measure.stderr)
if not blocks: raise RuntimeError(measure.stderr[-2000:])
stats=json.loads(blocks[-1])
flt=('loudnorm=I=-14:TP=-1:LRA=11:'
     f'measured_I={stats["input_i"]}:measured_LRA={stats["input_lra"]}:'
     f'measured_TP={stats["input_tp"]}:measured_thresh={stats["input_thresh"]}:'
     f'offset={stats["target_offset"]}:linear=true:print_format=summary')
subprocess.run(['ffmpeg','-y','-i',MIX_PRE,'-af',flt,'-t',str(duration),
                '-ar','48000','-ac','2','-c:a','pcm_s24le',MIX_MASTER],
               capture_output=True,text=True,check=True)
FINAL = f"{DIRS['delivery']}/PLYAZH_V2_MASTER_1080P24_RU.mp4"
subprocess.run(['ffmpeg','-y','-i',MASTER,'-i',MIX_MASTER,'-map','0:v','-map','1:a',
                '-c:v','copy','-c:a','aac','-b:a','256k','-ar','48000',
                '-t',str(duration),FINAL],capture_output=True,text=True,check=True)
if not os.path.exists(FINAL): raise RuntimeError('Финальный мастер не создан')
print('ГОТОВО:',FINAL,'| first-pass',stats)

### 14g. Титры и вшитые субтитры
Открывающий титр «Пляж», финальная карточка «Продолжение следует»
и версия мастера с вшитыми русскими субтитрами (чистая версия сохраняется отдельно).

In [ ]:
from PIL import Image, ImageDraw, ImageFont
import subprocess, os

# --- шрифт с кириллицей ---
!apt-get -qq install -y fonts-dejavu-core > /dev/null 2>&1
FONT = '/usr/share/fonts/truetype/dejavu/DejaVuSerif.ttf'
if not os.path.exists(FONT):
    FONT = '/usr/share/fonts/truetype/dejavu/DejaVuSans.ttf'

def card(text, path, size=96, sub=None):
    img = Image.new('RGBA', (1920,1080), (0,0,0,0))
    d = ImageDraw.Draw(img)
    f = ImageFont.truetype(FONT, size)
    bb = d.textbbox((0,0), text, font=f)
    x, y = (1920-(bb[2]-bb[0]))//2, (1080-(bb[3]-bb[1]))//2 - 20
    d.text((x+3,y+3), text, font=f, fill=(0,0,0,140))          # мягкая тень
    d.text((x,y), text, font=f, fill=(244,239,230,255))        # тёплый белый
    if sub:
        fs = ImageFont.truetype(FONT, size//3)
        bs = d.textbbox((0,0), sub, font=fs)
        d.text(((1920-(bs[2]-bs[0]))//2, y+size+30), sub, font=fs, fill=(200,195,185,235))
    img.save(path)
    return path

T_OPEN = card('Пляж', '/content/title_open.png', 120)
T_END  = card('Продолжение следует', '/content/title_end.png', 76)
print('Титры отрисованы')

if not os.path.exists(FINAL): raise FileNotFoundError(FINAL)
SRC = FINAL
dur = float(subprocess.run(['ffprobe','-v','error','-show_entries','format=duration',
                            '-of','csv=p=0', SRC], capture_output=True, text=True).stdout.strip())

# титр в начале (2.5-7.5 c) и в конце (последние 6 c), оба с плавным появлением
TITLED = f"{DIRS['delivery']}/PLYAZH_V2_TITLED_1080P24.mp4"
fc = ("[1:v]format=rgba,fade=t=in:st=2.5:d=1.2:alpha=1,"
      "fade=t=out:st=6.3:d=1.2:alpha=1[t1];"
      f"[2:v]format=rgba,fade=t=in:st={dur-6:.1f}:d=1.5:alpha=1[t2];"
      "[0:v][t1]overlay=0:0:enable='between(t,2.5,7.5)'[v1];"
      f"[v1][t2]overlay=0:0:enable='gte(t,{dur-6:.1f})'[v]")
r = subprocess.run(['ffmpeg','-y','-i',SRC,'-loop','1','-t',str(dur),'-i',T_OPEN,
                    '-loop','1','-t',str(dur),'-i',T_END,
                    '-filter_complex',fc,'-map','[v]','-map','0:a?',
                    '-c:v','libx264','-crf','18','-pix_fmt','yuv420p',
                    '-c:a','copy', TITLED], capture_output=True, text=True)
if r.returncode or not os.path.exists(TITLED): raise RuntimeError(r.stderr[-2000:])
print('С титрами:', TITLED)

# --- вшитые субтитры ---
SUBBED = f"{DIRS['delivery']}/PLYAZH_V2_MASTER_1080P24_RU_SUB.mp4"
if os.path.exists(SRT) and os.path.exists(TITLED):
    style = ("FontName=DejaVu Sans,FontSize=21,PrimaryColour=&H00E6EFF4,"
             "OutlineColour=&H90000000,BorderStyle=1,Outline=2,Shadow=1,"
             "MarginV=54,Alignment=2")
    r = subprocess.run(['ffmpeg','-y','-i',TITLED,
                        '-vf', f"subtitles='{SRT}':force_style='{style}'",
                        '-c:v','libx264','-crf','18','-pix_fmt','yuv420p',
                        '-c:a','copy', SUBBED], capture_output=True, text=True)
    if r.returncode or not os.path.exists(SUBBED): raise RuntimeError(r.stderr[-2000:])
    print('С субтитрами:', SUBBED)

print('\nЧистая версия без субтитров остаётся:', TITLED)

## 15. Финальный контроль
Жёстко проверяем 1920×1080, 24 fps, длительность ≥311.4 с, наличие звука,
громкость −14 LUFS и true peak не выше −1 dBFS; отдельно пишем black/freeze-отчёт.

In [ ]:
import subprocess, json, os, glob, re
from fractions import Fraction
target = f"{DIRS['delivery']}/PLYAZH_V2_TITLED_1080P24.mp4"
if not os.path.exists(target): raise FileNotFoundError(target)

info = json.loads(subprocess.run(
    ['ffprobe','-v','quiet','-print_format','json','-show_format','-show_streams', target],
    capture_output=True, text=True, check=True).stdout)

video=[s for s in info['streams'] if s['codec_type']=='video']
audio=[s for s in info['streams'] if s['codec_type']=='audio']
if len(video)!=1 or not audio: raise RuntimeError(f'Потоки video={len(video)} audio={len(audio)}')
v=video[0]; fps=float(Fraction(v['avg_frame_rate'])); dur=float(info['format']['duration'])
errors=[]
if (v['width'],v['height'])!=(1920,1080): errors.append(f"resolution {v['width']}x{v['height']}")
if abs(fps-24)>0.01: errors.append(f'fps {fps}')
if dur<311.4: errors.append(f'duration {dur}')
if errors: raise RuntimeError('QC specs: '+', '.join(errors))
for st in info['streams']:
    if st['codec_type']=='video':
        print(f"Видео: {st['width']}x{st['height']} {st.get('r_frame_rate')} "
              f"{st['codec_name']} {st.get('pix_fmt')}")
    else:
        print(f"Аудио: {st['codec_name']} {st.get('sample_rate')} Гц {st.get('channels')} кан.")
print(f"Длительность: {dur:.1f} c ({int(dur//60)}:{int(dur%60):02d})")
print(f"Размер: {int(info['format']['size'])/1024**2:.1f} МБ")

lm=subprocess.run(['ffmpeg','-hide_banner','-i',target,'-vn','-af',
                   'loudnorm=I=-14:TP=-1:LRA=11:print_format=json','-f','null','-'],
                  capture_output=True,text=True,check=True)
blocks=re.findall(r'\{[\s\S]*?\}',lm.stderr)
if not blocks: raise RuntimeError('Нет loudnorm-отчёта')
loud=json.loads(blocks[-1]); measured_lufs=float(loud['input_i']); measured_tp=float(loud['input_tp'])
if abs(measured_lufs+14)>0.35 or measured_tp>-0.8:
    raise RuntimeError(f'QC loudness: {measured_lufs} LUFS, {measured_tp} dBTP')
print(f'Громкость: {measured_lufs:.2f} LUFS | true peak {measured_tp:.2f} dBTP')

bl = subprocess.run(['ffmpeg','-i',target,'-vf','blackdetect=d=0.5:pic_th=0.98',
                     '-an','-f','null','-'], capture_output=True, text=True).stderr
black_hits=[l.strip() for l in bl.split('\n') if 'black_start:' in l]
fr = subprocess.run(['ffmpeg','-i',target,'-vf','freezedetect=n=-50dB:d=1.5',
                     '-an','-f','null','-'],capture_output=True,text=True).stderr
freeze_hits=[l.strip() for l in fr.split('\n') if 'freeze_start:' in l]
report={'target':target,'duration_sec':dur,'resolution':[v['width'],v['height']],
        'fps':fps,'loudness_lufs':measured_lufs,'true_peak_dbtp':measured_tp,
        'black_segments':black_hits,'freeze_events':freeze_hits,
        'face_motion_qc':f"{DIRS['clips']}/face_motion_qc.json"}
json.dump(report,open(f"{DIRS['delivery']}/QC_REPORT.json",'w'),ensure_ascii=False,indent=2)
print('Чёрные сегменты:',len(black_hits) or 'нет','| freeze-события:',len(freeze_hits) or 'нет')
print('QC PASS: 1080p24, полный хронометраж, звук −14 LUFS, face-motion QC')
print('\nФайлы на Drive:')
for f in sorted(glob.glob(f"{DIRS['delivery']}/*")):
    print(' ', os.path.basename(f), f'{os.path.getsize(f)/1024**2:.1f} МБ')

---
## Если что-то пошло не так

**Лица плохо похожи** → поднимите `weight` до 0.95 и `denoise` до 0.55 в ячейке 10,
используйте фото где лицо крупнее и резче, проверьте что antelopev2 распаковался.

**Лица Джона и Мэри смешиваются** → это лечится только раздельными проходами
(уже реализовано в `face_order`). Никогда не подавайте оба фото в один проход.

**Кончилась сессия Colab** → перезапустите ячейки 1–8, затем ту, на которой встали.
Готовые файлы на Drive пропускаются автоматически.

**LTX не влезает в память** → закройте просмотрщики, перезапустите runtime и продолжите
с кэша. Разрешение должно оставаться кратным 32; статичный fallback намеренно запрещён.

**Стиль недостаточно выражен** → `arcane_style_xl_v1.safetensors` уже подключена
через `LoraLoader`; поднимайте `ARCANE_WEIGHT` с 0.80 максимум до 0.90.